In [13]:
import os
import matplotlib.pyplot as plt
import csv
from PIL import Image
import numpy as np
import pickle
# Use appropriate regression metrics for multi-output
from sklearn.metrics import mean_squared_error, mean_absolute_error 
from skimage.color import rgb2gray
from skimage import exposure

In [14]:
X_COL, Y_COL, XW_COL, YW_COL = 1, 2, 3, 4 

# --- MODEL PATHS ---
RF_MODEL_PATH = 'saved_models/rf_model.sav' 
SVR_MODEL_PATH = 'saved_models/svr_model.sav' 
# --- OUTPUT FILENAMES ---
RF_OUTPUT_FILENAME = 'rf_prediction_results_test.csv'
SVR_OUTPUT_FILENAME = 'svr_prediction_results_test.csv'

## Defining Function

1. read_Image_and_CSV_Data() is used to read images and 4 outputs and pre-process them. It will return a list of images (NumPy arrays), NumPy array of corresponding outputs (N, 4)
2. save_CSV() is used to writes image names and 4 predicted outputs to a CSV file.

In [ ]:
def read_Image_and_CSV_Data(rootpath):

    images = [] 
    all_outputs = []
    nameList=[] 
    
    prefix = rootpath + '/' 
    
    try:
        gtFile = open(prefix + 'myData'+ '.csv') 
        gtReader = csv.reader(gtFile, delimiter=';') 
        next(gtReader) # Skip header row
        
        for row in gtReader:
            if not row or len(row) <= YW_COL:
                continue
            
            
            # Image Loading
            img_path = prefix + row[0]
            img = Image.open(img_path)

            # Resize to 32x32
            img = img.resize((32, 32), Image.BICUBIC) 
            img_array = np.array(img)

            # Convert to grayscale
            gray_img = rgb2gray(img_array)
            
            # Apply CLAHE (Contrast Enhancement)
            clache_img = exposure.equalize_adapthist(gray_img, clip_limit=0.03)
            
            images.append(clache_img)
            
            # Read all 4 outputs 
            x = int(row[X_COL])
            y = int(row[Y_COL])
            xw = int(row[XW_COL])
            yw = int(row[YW_COL])
            all_outputs.append([x, y, xw, yw]) 
            
            nameList.append(row[0]) 
            
        gtFile.close()
    
    except FileNotFoundError:
        print(f"Error: Test file not found at {prefix + 'myData.csv'}")
        exit()
    except Exception as e:
        print(f"Skipping row due to error: {e}")
        
    outputs_array = np.array(all_outputs)
    return images, outputs_array, nameList


def save_CSV(filename, name_list, y_predicted):

    # Write to CSV
    with open(filename, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["filename", "x_pred", "y_pred", "x_width_pred","y_width_pred"]) 
        
        for imagename, pred_row in zip(name_list, y_predicted):
            # *pred_row unpacks the 4 values: [x_pred, y_pred, x_width_pred, y_width_pred]
            writer.writerow([imagename, *pred_row])

## Main Execution Code

1. Load and Prepare Test Data

In [ ]:
testImages, testOutputs, TestNameList = read_Image_and_CSV_Data('Test')
print(f'Number of test data: {len(testOutputs)}')

X = []
Y = testOutputs.copy() # Y is the ground truth (N, 4)

for i in range(0,len(testOutputs)):
    # input X is the flattened image (32*32=1024 features)
    X.append(testImages[i].flatten()) 

X = np.array(X)
print(f'Shape of feature matrix X for test data: {X.shape}')

Number of test data: 40
Shape of feature matrix X for test data: (40, 1024)


2. Test Random Forest

In [17]:
print("\n--- Testing Random Forest Regressor ---")

try:
    rf_clf = pickle.load(open(RF_MODEL_PATH, 'rb'))
except FileNotFoundError:
    print(f"Error: RF model file '{RF_MODEL_PATH}' not found. Ensure trainer was run and model was saved.")
    exit()

# make prediction
Y_pred_rf = rf_clf.predict(X)

# check the accuracy
MSE_rf = mean_squared_error(Y, Y_pred_rf)
MAE_rf = mean_absolute_error(Y, Y_pred_rf)
print(f'Random Forest Test Mean Squared Error (MSE): {MSE_rf:.4f}')
print(f'Random Forest Test Mean Absolute Error (MAE): {MAE_rf:.4f}')

# save the csv file
save_CSV(RF_OUTPUT_FILENAME, TestNameList, Y_pred_rf)
print(f"Saved Random Forest prediction results to '{RF_OUTPUT_FILENAME}'")


--- Testing Random Forest Regressor ---
Random Forest Test Mean Squared Error (MSE): 1206.1646
Random Forest Test Mean Absolute Error (MAE): 26.8219
Saved Random Forest prediction results to 'rf_prediction_results_test.csv'


3. Test SVR

In [ ]:
print("\n--- Testing Support Vector Regressor (SVR) ---")

try:
    svr_clf = pickle.load(open(SVR_MODEL_PATH, 'rb'))
except FileNotFoundError:
    print(f"Error: SVR model file '{SVR_MODEL_PATH}' not found. Ensure trainer was run and model was saved.")
    exit()

# make prediction
Y_pred_svr = svr_clf.predict(X)

# check the accuracy
MSE_svr = mean_squared_error(Y, Y_pred_svr)
MAE_svr = mean_absolute_error(Y, Y_pred_svr)
print(f'SVR Test Mean Squared Error (MSE): {MSE_svr:.4f}')
print(f'SVR Test Mean Absolute Error (MAE): {MAE_svr:.4f}')

# save the csv file
save_CSV(SVR_OUTPUT_FILENAME, TestNameList, Y_pred_svr)
print(f"Saved SVR prediction results to '{SVR_OUTPUT_FILENAME}'")


--- Testing Support Vector Regressor (SVR) ---
SVR Test Mean Squared Error (MSE): 1121.0041
SVR Test Mean Absolute Error (MAE): 25.1874
Saved SVR prediction results to 'svr_prediction_results_test.csv'
